In [24]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from scipy.ndimage import label
from sklearn import metrics
from sklearn.metrics import precision_score, recall_score, accuracy_score, f1_score
from itertools import combinations
from collections import Counter

Read in the csv data

In [9]:
df = pd.read_csv("C:\\Users\\Tony\\Documents\\GitHub\\EAE-598-Project\\final_all_events.csv")
df.head()

,time,level,pv_300,pv_700,pv_850,pv_925,pv_1000,wnd_300,wnd_500,wnd_850,...,thetae_grad_850,thetae_grad_925,thetae_grad_1000,t_grad_850,t_grad_925,t_grad_1000,label,year,month,day
0,2017-02-20 03:00:00,300.0,1.615085e-12,5.168937e-13,1.192982e-12,8.204532e-13,3.146415e-13,66.406730,46.775124,14.784239,...,12.238490,9.950269,7.297508,5.513050,3.916206,2.792917,MFW,2017,2,20
1,2019-02-13 17:00:00,300.0,2.824925e-12,6.330504e-13,7.191460e-13,2.770781e-13,1.959496e-13,62.199850,33.380860,4.886407,...,9.568939,9.987158,5.880394,3.667785,3.668505,1.591186,MFW,2019,2,13
2,2015-11-19 02:00:00,300.0,-3.753535e-14,5.650719e-13,7.535887e-13,1.251950e-12,1.095274e-13,46.878395,32.892265,23.937796,...,4.126061,4.966980,6.084853,1.439914,1.810055,2.012980,MFW,2015,11,19
3,2016-01-28 17:00:00,300.0,1.088309e-13,8.542314e-13,2.183340e-12,1.469326e-12,4.105159e-13,62.548008,47.574287,23.934720,...,5.187433,5.209870,2.882797,3.458344,1.929978,0.705986,MFW,2016,1,28
4,2014-02-07 13:00:00,300.0,3.312839e-13,5.557943e-13,8.091834e-13,2.026526e-12,7.450257e-13,43.977250,34.692270,24.003717,...,6.107335,8.828790,3.782991,0.839231,2.512393,1.679689,MFW,2014,2,7


Calculate additional variables 

In [10]:
df['vort_def'] =  df['rel_vort_1000'] / df['total_deformation_1000']
df['low_level_pv'] = (df['pv_925'] + df['pv_850']) / 2

Split the data into three subsets for training, validating, and testing. 

In [ ]:
# Random seed
zid = 1895595

# Splt the data 
train_val_df, test_df = train_test_split(df, test_size=0.2, random_state=zid)
train_df, val_df = train_test_split(train_val_df, test_size=0.125, random_state=zid)

# Print the shapes of the subsets
print("Training set shape:", train_df.shape)
print("Validation set shape:", val_df.shape)
print("Testing set shape:", test_df.shape)

Training set shape: (35, 68)
Validation set shape: (5, 68)
Testing set shape: (10, 68)


Define different model configurations

In [32]:
configurations = [
    # Base 
    {"n_estimators": 100, "max_depth": None, "min_samples_split": 2, "min_samples_leaf": 1},
    
    # Varying n_estimators
    {"n_estimators": 50, "max_depth": None, "min_samples_split": 2, "min_samples_leaf": 1},
    {"n_estimators": 200, "max_depth": None, "min_samples_split": 2, "min_samples_leaf": 1},
    {"n_estimators": 300, "max_depth": None, "min_samples_split": 2, "min_samples_leaf": 1},
    {"n_estimators": 400, "max_depth": None, "min_samples_split": 2, "min_samples_leaf": 1},
    {"n_estimators": 500, "max_depth": None, "min_samples_split": 2, "min_samples_leaf": 1},

    # Varying max_depth
    {"n_estimators": 100, "max_depth": 5, "min_samples_split": 2, "min_samples_leaf": 1},
    {"n_estimators": 100, "max_depth": 10, "min_samples_split": 2, "min_samples_leaf": 1},
    {"n_estimators": 100, "max_depth": 15, "min_samples_split": 2, "min_samples_leaf": 1},
    {"n_estimators": 100, "max_depth": 20, "min_samples_split": 2, "min_samples_leaf": 1},
    {"n_estimators": 100, "max_depth": 30, "min_samples_split": 2, "min_samples_leaf": 1},

    # Varying min_samples_split
    {"n_estimators": 100, "max_depth": None, "min_samples_split": 5, "min_samples_leaf": 1},
    {"n_estimators": 100, "max_depth": None, "min_samples_split": 10, "min_samples_leaf": 1},
    {"n_estimators": 100, "max_depth": None, "min_samples_split": 20, "min_samples_leaf": 1},
    {"n_estimators": 100, "max_depth": None, "min_samples_split": 50, "min_samples_leaf": 1},
    {"n_estimators": 100, "max_depth": None, "min_samples_split": 100, "min_samples_leaf": 1},
    
    # Varying min_samples_leaf
    {"n_estimators": 100, "max_depth": None, "min_samples_split": 2, "min_samples_leaf": 2},
    {"n_estimators": 100, "max_depth": None, "min_samples_split": 2, "min_samples_leaf": 5},
    {"n_estimators": 100, "max_depth": None, "min_samples_split": 2, "min_samples_leaf": 10},
    {"n_estimators": 100, "max_depth": None, "min_samples_split": 2, "min_samples_leaf": 15},
    {"n_estimators": 100, "max_depth": None, "min_samples_split": 2, "min_samples_leaf": 20},

    # Varying n_estimators and max_depth
    {"n_estimators": 500, "max_depth": 50, "min_samples_split": 2, "min_samples_leaf": 1},
    {"n_estimators": 1000, "max_depth": None, "min_samples_split": 2, "min_samples_leaf": 1},
    {"n_estimators": 150, "max_depth": 5, "min_samples_split": 2, "min_samples_leaf": 1},
    {"n_estimators": 200, "max_depth": 10, "min_samples_split": 2, "min_samples_leaf": 1},
    {"n_estimators": 300, "max_depth": 15, "min_samples_split": 2, "min_samples_leaf": 1},
    
    # Varying everything
    {"n_estimators": 100, "max_depth": 3, "min_samples_split": 20, "min_samples_leaf": 20},
    {"n_estimators": 150, "max_depth": 4, "min_samples_split": 30, "min_samples_leaf": 15},
    {"n_estimators": 300, "max_depth": 15, "min_samples_split": 8, "min_samples_leaf": 4},
    {"n_estimators": 250, "max_depth": 12, "min_samples_split": 6, "min_samples_leaf": 3},
    {"n_estimators": 200, "max_depth": 10, "min_samples_split": 4, "min_samples_leaf": 2},
    {"n_estimators": 400, "max_depth": 25, "min_samples_split": 15, "min_samples_leaf": 10},
    {"n_estimators": 350, "max_depth": 30, "min_samples_split": 20, "min_samples_leaf": 5},
    {"n_estimators": 600, "max_depth": 6, "min_samples_split": 5, "min_samples_leaf": 5},
    {"n_estimators": 80, "max_depth": 4, "min_samples_split": 10, "min_samples_leaf": 5},
    {"n_estimators": 60, "max_depth": 2, "min_samples_split": 2, "min_samples_leaf": 1},
    {"n_estimators": 60, "max_depth": 3, "min_samples_split": 4, "min_samples_leaf": 2},
    {"n_estimators": 120, "max_depth": 13, "min_samples_split": 13, "min_samples_leaf": 13},
    {"n_estimators": 175, "max_depth": 17, "min_samples_split": 7, "min_samples_leaf": 3},
    {"n_estimators": 220, "max_depth": 22, "min_samples_split": 11, "min_samples_leaf": 7},
    {"n_estimators": 800, "max_depth": None, "min_samples_split": 5, "min_samples_leaf": 1},
    {"n_estimators": 1000, "max_depth": 40, "min_samples_split": 10, "min_samples_leaf": 2},
    {"n_estimators": 200, "max_depth": 10, "min_samples_split": 5, "min_samples_leaf": 2},
    {"n_estimators": 300, "max_depth": 20, "min_samples_split": 10, "min_samples_leaf": 5},
    {"n_estimators": 50, "max_depth": 5, "min_samples_split": 20, "min_samples_leaf": 10}]

Defined function to test the different model configurations

In [ ]:
def evaluate_rf_with_configurations(train_df, val_df, var1, var2, var3, label_col, configurations, random_state=1895595):
    """
    Train and evaluate Random Forest models with multiple configurations.

    Parameters
    ----------
    train_df : pd.DataFrame
        Training dataset.
    val_df : pd.DataFrame
        Validation dataset.
    var1 : str
        Name of the first variable.
    var2 : str
        Name of the second variable.
    var3 : str
        Name of the third variable.
    label_col : str
        Name of the label column.
    configurations : list
        List of hyperparameter configurations.
    random_state : int, optional
        Random state for reproducibility (default is 1895595).

    Returns
    -------
    best_config : dict
        Best hyperparameter configuration.
    best_metrics : dict
        Metrics of the best model, including accuracy, precision, recall, and F1-score.
    best_model : RandomForestClassifier
        Best trained Random Forest model.

    """
    
    best_model = None
    best_metrics = {"accuracy": 0, "precision": 0, "recall": 0, "f1_score": 0}
    best_config = None

    for config in configurations:
        print(f"Testing configuration: {config}")
        rf_clf = RandomForestClassifier(
            n_estimators=config["n_estimators"],
            max_depth=config["max_depth"],
            min_samples_split=config["min_samples_split"],
            min_samples_leaf=config["min_samples_leaf"],
            random_state=random_state
        )

        # Train the model
        rf_clf.fit(train_df[[var1, var2, var3]].values, train_df[label_col].values)

        # Predict on validation data
        predicted = rf_clf.predict(val_df[[var1, var2, var3]].values)
        expected = val_df[label_col].values

        # Calculate metrics
        accuracy = accuracy_score(expected, predicted)
        precision = precision_score(expected, predicted, average="weighted")
        recall = recall_score(expected, predicted, average="weighted")
        f1 = f1_score(expected, predicted, average="weighted")

        #print(f"Metrics: Accuracy={accuracy}, Precision={precision}, Recall={recall}, F1-Score={f1}")

        # Update the best model
        if f1 > best_metrics["f1_score"]:
            best_model = rf_clf
            best_metrics = {"accuracy": accuracy, "precision": precision, "recall": recall, "f1_score": f1}
            best_config = config

    print("Best Configuration:")
    print(best_config)
    print("Best Metrics:")
    print(best_metrics)

    return best_config, best_metrics, best_model

Let's do some testing with different variable combinations

In [ ]:
best_config, best_metrics, best_model = evaluate_rf_with_configurations(
    train_df=train_df,
    val_df=val_df,
    var1="pv_925",
    var2="ivt",
    var3="t_grad_850",
    label_col="label",
    configurations=configurations,
    random_state=zid)

Testing configuration: {'n_estimators': 100, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 50, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 200, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 300, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 400, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 500, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 100, 'max_depth': 5, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 100, 'max_depth': 10, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 100, 'max_depth': 15, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing c

C:\Users\Tony\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Tony\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Testing configuration: {'n_estimators': 100, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 2}
Testing configuration: {'n_estimators': 100, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 5}
Testing configuration: {'n_estimators': 100, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 10}
Testing configuration: {'n_estimators': 100, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 15}
Testing configuration: {'n_estimators': 100, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 20}
Testing configuration: {'n_estimators': 500, 'max_depth': 50, 'min_samples_split': 2, 'min_samples_leaf': 1}


C:\Users\Tony\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Tony\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Testing configuration: {'n_estimators': 1000, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 150, 'max_depth': 5, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 200, 'max_depth': 10, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 300, 'max_depth': 15, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 100, 'max_depth': 3, 'min_samples_split': 20, 'min_samples_leaf': 20}
Testing configuration: {'n_estimators': 150, 'max_depth': 4, 'min_samples_split': 30, 'min_samples_leaf': 15}


C:\Users\Tony\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Tony\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Testing configuration: {'n_estimators': 300, 'max_depth': 15, 'min_samples_split': 8, 'min_samples_leaf': 4}
Testing configuration: {'n_estimators': 250, 'max_depth': 12, 'min_samples_split': 6, 'min_samples_leaf': 3}
Testing configuration: {'n_estimators': 200, 'max_depth': 10, 'min_samples_split': 4, 'min_samples_leaf': 2}
Testing configuration: {'n_estimators': 400, 'max_depth': 25, 'min_samples_split': 15, 'min_samples_leaf': 10}
Testing configuration: {'n_estimators': 350, 'max_depth': 30, 'min_samples_split': 20, 'min_samples_leaf': 5}
Testing configuration: {'n_estimators': 600, 'max_depth': 6, 'min_samples_split': 5, 'min_samples_leaf': 5}
Testing configuration: {'n_estimators': 80, 'max_depth': 4, 'min_samples_split': 10, 'min_samples_leaf': 5}
Testing configuration: {'n_estimators': 60, 'max_depth': 2, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 60, 'max_depth': 3, 'min_samples_split': 4, 'min_samples_leaf': 2}
Testing configuration:

C:\Users\Tony\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Testing configuration: {'n_estimators': 220, 'max_depth': 22, 'min_samples_split': 11, 'min_samples_leaf': 7}
Testing configuration: {'n_estimators': 800, 'max_depth': None, 'min_samples_split': 5, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 1000, 'max_depth': 40, 'min_samples_split': 10, 'min_samples_leaf': 2}
Testing configuration: {'n_estimators': 200, 'max_depth': 10, 'min_samples_split': 5, 'min_samples_leaf': 2}
Testing configuration: {'n_estimators': 300, 'max_depth': 20, 'min_samples_split': 10, 'min_samples_leaf': 5}
Testing configuration: {'n_estimators': 50, 'max_depth': 5, 'min_samples_split': 20, 'min_samples_leaf': 10}
Best Configuration:
{'n_estimators': 100, 'max_depth': None, 'min_samples_split': 10, 'min_samples_leaf': 1}
Best Metrics:
{'accuracy': 0.6, 'precision': 0.6, 'recall': 0.6, 'f1_score': 0.6}


Another

In [ ]:
best_config, best_metrics, best_model = evaluate_rf_with_configurations(
    train_df=train_df,
    val_df=val_df,
    var1="pv_925",
    var2="ivt",
    var3="tadv_925",
    label_col="label",
    configurations=configurations,
    random_state=zid)

Testing configuration: {'n_estimators': 100, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 50, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 200, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 300, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 400, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 500, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 100, 'max_depth': 5, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 100, 'max_depth': 10, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 100, 'max_depth': 15, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing c

C:\Users\Tony\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Tony\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Testing configuration: {'n_estimators': 100, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 5}
Testing configuration: {'n_estimators': 100, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 10}
Testing configuration: {'n_estimators': 100, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 15}
Testing configuration: {'n_estimators': 100, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 20}


C:\Users\Tony\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Tony\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Testing configuration: {'n_estimators': 500, 'max_depth': 50, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 1000, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 150, 'max_depth': 5, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 200, 'max_depth': 10, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 300, 'max_depth': 15, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 100, 'max_depth': 3, 'min_samples_split': 20, 'min_samples_leaf': 20}
Testing configuration: {'n_estimators': 150, 'max_depth': 4, 'min_samples_split': 30, 'min_samples_leaf': 15}


...

Testing configuration: {'n_estimators': 300, 'max_depth': 15, 'min_samples_split': 8, 'min_samples_leaf': 4}
Testing configuration: {'n_estimators': 250, 'max_depth': 12, 'min_samples_split': 6, 'min_samples_leaf': 3}
Testing configuration: {'n_estimators': 200, 'max_depth': 10, 'min_samples_split': 4, 'min_samples_leaf': 2}


C:\Users\Tony\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Testing configuration: {'n_estimators': 400, 'max_depth': 25, 'min_samples_split': 15, 'min_samples_leaf': 10}
Testing configuration: {'n_estimators': 350, 'max_depth': 30, 'min_samples_split': 20, 'min_samples_leaf': 5}
Testing configuration: {'n_estimators': 600, 'max_depth': 6, 'min_samples_split': 5, 'min_samples_leaf': 5}
Testing configuration: {'n_estimators': 80, 'max_depth': 4, 'min_samples_split': 10, 'min_samples_leaf': 5}
Testing configuration: {'n_estimators': 60, 'max_depth': 2, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 60, 'max_depth': 3, 'min_samples_split': 4, 'min_samples_leaf': 2}
Testing configuration: {'n_estimators': 120, 'max_depth': 13, 'min_samples_split': 13, 'min_samples_leaf': 13}


C:\Users\Tony\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Tony\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Tony\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Testing configuration: {'n_estimators': 175, 'max_depth': 17, 'min_samples_split': 7, 'min_samples_leaf': 3}
Testing configuration: {'n_estimators': 220, 'max_depth': 22, 'min_samples_split': 11, 'min_samples_leaf': 7}
Testing configuration: {'n_estimators': 800, 'max_depth': None, 'min_samples_split': 5, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 1000, 'max_depth': 40, 'min_samples_split': 10, 'min_samples_leaf': 2}
Testing configuration: {'n_estimators': 200, 'max_depth': 10, 'min_samples_split': 5, 'min_samples_leaf': 2}
Testing configuration: {'n_estimators': 300, 'max_depth': 20, 'min_samples_split': 10, 'min_samples_leaf': 5}
Testing configuration: {'n_estimators': 50, 'max_depth': 5, 'min_samples_split': 20, 'min_samples_leaf': 10}
Best Configuration:
{'n_estimators': 100, 'max_depth': None, 'min_samples_split': 10, 'min_samples_leaf': 1}
Best Metrics:
{'accuracy': 0.8, 'precision': 0.8666666666666666, 'recall': 0.8, 'f1_score': 0.8}


Last

In [ ]:
best_config, best_metrics, best_model = evaluate_rf_with_configurations(
    train_df=train_df,
    val_df=val_df,
    var1="pv_925",
    var2="ivt",
    var3="z_1000",
    label_col="label",
    configurations=configurations,
    random_state=zid)

Testing configuration: {'n_estimators': 100, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 50, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 200, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 300, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 400, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 500, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 100, 'max_depth': 5, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 100, 'max_depth': 10, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 100, 'max_depth': 15, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing c

C:\Users\Tony\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Tony\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Testing configuration: {'n_estimators': 100, 'max_depth': None, 'min_samples_split': 100, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 100, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 2}
Testing configuration: {'n_estimators': 100, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 5}
Testing configuration: {'n_estimators': 100, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 10}
Testing configuration: {'n_estimators': 100, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 15}
Testing configuration: {'n_estimators': 100, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 20}


C:\Users\Tony\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Tony\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Testing configuration: {'n_estimators': 500, 'max_depth': 50, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 1000, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 150, 'max_depth': 5, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 200, 'max_depth': 10, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 300, 'max_depth': 15, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 100, 'max_depth': 3, 'min_samples_split': 20, 'min_samples_leaf': 20}
Testing configuration: {'n_estimators': 150, 'max_depth': 4, 'min_samples_split': 30, 'min_samples_leaf': 15}


C:\Users\Tony\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\Tony\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Testing configuration: {'n_estimators': 300, 'max_depth': 15, 'min_samples_split': 8, 'min_samples_leaf': 4}
Testing configuration: {'n_estimators': 250, 'max_depth': 12, 'min_samples_split': 6, 'min_samples_leaf': 3}
Testing configuration: {'n_estimators': 200, 'max_depth': 10, 'min_samples_split': 4, 'min_samples_leaf': 2}
Testing configuration: {'n_estimators': 400, 'max_depth': 25, 'min_samples_split': 15, 'min_samples_leaf': 10}
Testing configuration: {'n_estimators': 350, 'max_depth': 30, 'min_samples_split': 20, 'min_samples_leaf': 5}
Testing configuration: {'n_estimators': 600, 'max_depth': 6, 'min_samples_split': 5, 'min_samples_leaf': 5}
Testing configuration: {'n_estimators': 80, 'max_depth': 4, 'min_samples_split': 10, 'min_samples_leaf': 5}
Testing configuration: {'n_estimators': 60, 'max_depth': 2, 'min_samples_split': 2, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 60, 'max_depth': 3, 'min_samples_split': 4, 'min_samples_leaf': 2}
Testing configuration:

C:\Users\Tony\anaconda3\lib\site-packages\sklearn\metrics\_classification.py:1509: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Testing configuration: {'n_estimators': 175, 'max_depth': 17, 'min_samples_split': 7, 'min_samples_leaf': 3}
Testing configuration: {'n_estimators': 220, 'max_depth': 22, 'min_samples_split': 11, 'min_samples_leaf': 7}
Testing configuration: {'n_estimators': 800, 'max_depth': None, 'min_samples_split': 5, 'min_samples_leaf': 1}
Testing configuration: {'n_estimators': 1000, 'max_depth': 40, 'min_samples_split': 10, 'min_samples_leaf': 2}
Testing configuration: {'n_estimators': 200, 'max_depth': 10, 'min_samples_split': 5, 'min_samples_leaf': 2}
Testing configuration: {'n_estimators': 300, 'max_depth': 20, 'min_samples_split': 10, 'min_samples_leaf': 5}
Testing configuration: {'n_estimators': 50, 'max_depth': 5, 'min_samples_split': 20, 'min_samples_leaf': 10}
Best Configuration:
{'n_estimators': 100, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1}
Best Metrics:
{'accuracy': 0.8, 'precision': 0.85, 'recall': 0.8, 'f1_score': 0.7809523809523808}


Okay, let's expand upon the previous defined function and loop over numerous variable combinations to see which configuration is best

In [ ]:
def evaluate_rf_with_configurations_loop(train_df, val_df, variables, label_col, configurations, random_state=1895595):
    """
    Train and evaluate Random Forest models with multiple configurations and variable combinations.

    Parameters
    ----------
    train_df : pd.DataFrame
        Training dataset.
    val_df : pd.DataFrame
        Validation dataset.
    variables : list
        List of variable names to test in combinations of three.
    label_col : str
        Name of the label column.
    configurations : list
        List of hyperparameter configurations.
    random_state : int, optional
        Random state for reproducibility (default is 1895595).

    Returns
    -------
    best_combination : tuple
        Best combination of three variables.
    best_config : dict
        Best hyperparameter configuration.
    best_metrics : dict
        Metrics of the best model, including accuracy, precision, recall, and F1-score.
    best_model : RandomForestClassifier
        Best trained Random Forest model.

    """

    best_configurations = []

    # Loop through all combinations
    for combination in combinations(variables, 3):
        var1, var2, var3 = combination
        #print(f"Testing combination: {var1}, {var2}, {var3}")

        best_f1 = 0
        best_config = None

        # Test all configurations for the current combination
        for config in configurations:
            #print(f"Testing configuration: {config}")
            rf_clf = RandomForestClassifier(
                n_estimators=config["n_estimators"],
                max_depth=config["max_depth"],
                min_samples_split=config["min_samples_split"],
                min_samples_leaf=config["min_samples_leaf"],
                random_state=random_state)

            # Train the model
            rf_clf.fit(train_df[[var1, var2, var3]].values, train_df[label_col].values)

            # Predict on validation data
            predicted = rf_clf.predict(val_df[[var1, var2, var3]].values)
            expected = val_df[label_col].values

            # Calculate F1-score
            f1 = f1_score(expected, predicted, average="weighted")

            # Update the best configuration for this combination
            if f1 > best_f1:
                best_f1 = f1
                best_config = config

        # Track the best configuration for this combination
        if best_config:
            best_configurations.append(tuple(best_config.items()))

    # Count the occurrences of each configuration
    config_counts = Counter(best_configurations)

    # Find the most common configuration
    most_common_config = dict(config_counts.most_common(1)[0][0])

    print("Most Common Best Configuration:")
    print(most_common_config)
    print("Configuration Counts:")
    print(config_counts)

    return most_common_config, config_counts

In [ ]:
variable_list = ["pv_925", "ivt", "t_grad_850", "z_1000", "tadv_925", "rel_vort_1000"]

most_common_config, config_counts = evaluate_rf_with_configurations_loop(
    train_df=train_df,
    val_df=val_df,
    variables=variable_list,
    label_col="label",
    configurations=configurations,
    random_state=zid)

Most Common Best Configuration:
{'n_estimators': 100, 'max_depth': None, 'min_samples_split': 2, 'min_samples_leaf': 1}
Configuration Counts:
Counter({(('n_estimators', 100), ('max_depth', None), ('min_samples_split', 2), ('min_samples_leaf', 1)): 10, (('n_estimators', 100), ('max_depth', None), ('min_samples_split', 2), ('min_samples_leaf', 5)): 4, (('n_estimators', 100), ('max_depth', None), ('min_samples_split', 10), ('min_samples_leaf', 1)): 3, (('n_estimators', 100), ('max_depth', None), ('min_samples_split', 50), ('min_samples_leaf', 1)): 1, (('n_estimators', 400), ('max_depth', None), ('min_samples_split', 2), ('min_samples_leaf', 1)): 1, (('n_estimators', 50), ('max_depth', None), ('min_samples_split', 2), ('min_samples_leaf', 1)): 1})


So the most common best model configuration is one that has 100 n_estimators, no max_depth, 2 min_samples_split, and 1 min_samples_leaf